# 因子框架原理和细节

本篇供排查数据读写、开发算子和扩展数据库时查阅。日常使用先看 [快速入门](QuickStart.ipynb)、[因子开发](因子开发.ipynb) 和 [HDF5DB](HDF5DB.ipynb)。运行身份、上下文隔离与跨轮缓存统一见 [Core 原理和细节](../Core/原理和细节.ipynb)。

<a id="hdf5-storage"></a>

## HDF5 存储布局

## 存储结构

HDF5DB 的逻辑层与存储层的对应关系：

```
MainDir/                     ← 因子库
  ├── table_1/               ← 因子表（一个子目录）
  │     ├── factor_a.hdf5    ← 因子（一个 HDF5 文件）
  │     ├── factor_b.hdf5
  │     ├── _TableInfo.h5    ← 表的元信息
  │     └── _Table.lock      ← 表锁
  ├── table_2/
  │     └── ...
  └── _FDB.lock              ← 库锁
```

每个 HDF5 因子文件内部包含三个 Dataset：

| Dataset | 内容 | 说明 |
|---------|------|------|
| `ID` | 证券代码序列 | shape=(n_ids,), dtype=String, utf-8 编码 |
| `DateTime` | 时点序列 | shape=(n_dts,), dtype=float, 时点转为 timestamp 存储 |
| `Data` | 因子数据矩阵 | shape=(n_dts, n_ids), double 存为 float64, string 存为 String, object 存为 vlen_dtype(uint8) |

因子的元数据存储在 HDF5 文件 root group 的 attrs 中，其中必须包含 `DataType` 属性。

### 锁机制

HDF5DB 使用文件锁保证并发安全：
- **库锁**（`_FDB.lock`）：创建表目录、重命名/删除表或因子时获取
- **表锁**（`_Table.lock`）：读写表级元信息时获取
- **因子锁**（`<factor_name>.lock`）：读写因子数据或元信息时需要获取

```mermaid
graph TD
    subgraph 逻辑层
        A[因子库<br/>HDF5DB]
        B1[因子表<br/>HDF5FactorTable]
        B2[因子表]
        A --> B1
        A --> B2
        F1[因子<br/>Factor]
        F2[因子]
        B1 --> F1
        B1 --> F2
    end

    subgraph 存储层
        C[主目录<br/>MainDir]
        D1[子目录<br/>table_1]
        D2[子目录<br/>table_2]
        C --> D1
        C --> D2
        E1[HDF5文件<br/>factor_a.hdf5]
        E2[HDF5文件<br/>factor_b.hdf5]
        D2 --> E1
        D2 --> E2
    end

    subgraph HDF5内部
        DS1[DateTime Dataset]
        DS2[ID Dataset]
        DS3[Data Dataset]
    end

    A -.-> C
    B1 -.-> D2
    F1 -.-> E2
    E2 -.-> HDF5内部

    style A fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style B1 fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style B2 fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style F1 fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style F2 fill:#c41e3a,color:#fff,stroke:#8b0000,stroke-width:2px
    style C fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
    style D1 fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
    style D2 fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
    style E1 fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
    style E2 fill:#d2b48c,color:#333,stroke:#8b7355,stroke-width:2px
    style DS1 fill:#ff8c00,color:#fff,stroke:#cc7000,stroke-width:2px
    style DS2 fill:#ff8c00,color:#fff,stroke:#cc7000,stroke-width:2px
    style DS3 fill:#ff8c00,color:#fff,stroke:#cc7000,stroke-width:2px
```

<a id="sql-metadata"></a>

# 元数据侧表

SQLDB 在数据库中维护了一张名为 `qs_meta` 的元数据侧表（可通过 `MetaTableName` 参数自定义表名），用于持久化存储因子表和因子的自定义元信息。

**侧表结构：**

| 列名 | 说明 |
|------|------|
| `table_name` | 因子表名称 |
| `field_name` | 因子名称（空字符串表示表级元数据） |
| `meta_key` | 元信息键 |
| `meta_value` | 元信息值 |

**设计要点：**

- 表级元数据：`field_name` 为空字符串，通过 `setTableMetaData` 写入
- 因子级元数据：`field_name` 为因子名，通过 `setFactorMetaData` 写入
- **级联更新**：重命名表/因子或删除表/因子时，侧表中的相关记录会自动同步更新或删除
- 侧表在 `connect()` 时自动创建（如果不存在），在 `connect()` 时自动加载元数据到 `_TableInfo` 和 `_FactorInfo`

元数据侧表使得 SQLDB 可以存储数据库 schema 之外的扩展信息（如因子描述、数据来源、更新频率等），而无需修改数据库表结构。

<a id="operator-binding"></a>

# 因子算子

## 模型参数和公开输入

所有因子算子继承 `FactorOperator`。计算参数仍在 `__QS_ArgClass__` 中声明，模型参数使用 `OperatorModelArgs` 子类，保留 Pydantic 的类型、默认值和取值范围。`OperatorType`、`DataType`、`CompoundType` 等字段决定计算与输出形式。

公开因子输入由 `__QS_InputClass__` 声明，模型继承 `FactorInputModel`。必填性来自默认值，多个因子使用 `tuple[Factor, ...]`，数量约束使用 `Field(min_length=..., max_length=...)`。`op.input_info()` 返回声明，绑定算子的查询还包括当前已固定的输入。`OperandInfo` 不再是算子参数字段；数值内核所需的槽位视图从输入模型生成。

`ToList` 的输入为因子集合 `x` 和可选掩码 `mask`；成分聚合的 `component_data` 与 `exog` 是独立集合。公开接口在展开前校验各组，最终因子保留命名依赖，输入是否存在和分组数量从依赖读取，不再在因子 `ModelArgs` 中重复保存。

## 部分绑定

`op.bind(**inputs)` 始终返回新算子，即使已经指定全部输入也不创建因子。直接调用算子始终返回因子；合并预绑定和本次输入后执行完整校验，缺失输入时报错。固定槽位在调用中不能覆盖，替换使用 `bind()`，重新开放使用 `unbind()`。

固定位置参数按剩余槽位的声明顺序分配。集合的 `bind()` 是整体替换，调用中的集合成员追加到预绑定成员之后；多个集合组建议通过名称传入。`new(args=...)` 修改模型配置并保留绑定，输入节点引用共享，算子参数容器与原算子隔离。

函数工厂和装饰器使用 `input_model=输入模型类`。Python 进程内序列化可保留绑定及动态模型；面向 GUI 的可持久化图协议由 QSExt 提供，不直接交换活动节点或 Python 序列化字节。
